# Nettoyage

Produire un fichier propre dans `data/processed/` à partir du CSV brut : renommage, dates, contrôles qualité, export.

## 1. Chargement et renommage

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/gcp_final_approved_dataset.csv")

renommage = {
    "Resource ID": "resource_id",
    "Service Name": "service",
    "Usage Quantity": "usage_quantity",
    "Usage Unit": "usage_unit",
    "Region/Zone": "region",
    "CPU Utilization (%)": "cpu_util_pct",
    "Memory Utilization (%)": "memory_util_pct",
    "Network Inbound Data (Bytes)": "net_in_bytes",
    "Network Outbound Data (Bytes)": "net_out_bytes",
    "Usage Start Date": "usage_start",
    "Usage End Date": "usage_end",
    "Cost per Quantity ($)": "cost_per_quantity",
    "Unrounded Cost ($)": "cost_usd",
    "Rounded Cost ($)": "cost_rounded",
    "Total Cost (INR)": "cost_inr",
}
df = df.rename(columns=renommage)
df.head()

,resource_id,service,usage_quantity,usage_unit,region,cpu_util_pct,memory_util_pct,net_in_bytes,net_out_bytes,usage_start,usage_end,cost_per_quantity,cost_usd,cost_rounded,cost_inr
0,res-ST6BAJ2N,Cloud Dataproc,954.9843,Requests,europe-north1,92.72,70.42,77097035547,8.268593e+10,01-08-2024 22:24,07-08-2024 06:54,5.24,5004.12,5004,415332
1,res-TYGNR0RV,Pub/Sub,479.6348,GB,europe-west1,37.78,46.80,8239357017,1.135149e+10,23-08-2024 09:18,25-08-2024 07:26,9.82,4710.01,4710,390930
2,res-S3I9C869,BigQuery,114.4129,GB,southamerica-east1,82.54,59.47,23358691883,2.345081e+10,18-07-2024 11:13,22-07-2024 00:45,7.37,843.22,843,69969
3,res-1RY9BZ6G,Cloud Endpoints,413.1930,GB,us-central1,86.68,97.63,53404385778,6.078019e+10,20-07-2024 00:15,21-07-2024 10:16,4.41,1822.18,1822,151226
4,res-S3HQYIZ2,Cloud Spanner,335.9892,GB,asia-southeast1,82.95,21.58,10795270461,1.416138e+10,21-08-2024 21:25,25-08-2024 00:21,6.09,2046.17,2046,169818


## 2. Dates

In [2]:
date_fmt = "%d-%m-%Y %H:%M"
df["usage_start"] = pd.to_datetime(df["usage_start"], format=date_fmt)
df["usage_end"] = pd.to_datetime(df["usage_end"], format=date_fmt)

df[["usage_start", "usage_end"]].dtypes

usage_start    datetime64[us]
usage_end      datetime64[us]
dtype: object

## 3. Durée d'usage

In [3]:
df["duree_jours"] = (df["usage_end"] - df["usage_start"]).dt.total_seconds() / 86400

print("durées <= 0 :", len(df[df["duree_jours"] <= 0]))
print("période couverte :", df["usage_start"].min(), "-", df["usage_end"].max())
df["duree_jours"].describe()

durées <= 0 : 0
période couverte : 2024-06-30 16:20:00 - 2024-09-03 14:19:00


count    1000.000000
mean        3.504212
std         1.437085
min         1.000694
25%         2.228472
50%         3.557292
75%         4.746007
max         5.999306
Name: duree_jours, dtype: float64

## 4. Contrôles de cohérence

In [4]:
# doublons
print("lignes dupliquées :", len(df[df.duplicated()]))
print("resource_id dupliqué :", len(df[df["resource_id"].duplicated()]))

# coûts et quantités
print("cost_usd <= 0 :", len(df[df["cost_usd"] <= 0]))
print("usage_quantity <= 0 :", len(df[df["usage_quantity"] <= 0]))

# pourcentages impossibles (négatifs ou > 100)
print("cpu_util_pct invalide :", len(df[(df["cpu_util_pct"] < 0) | (df["cpu_util_pct"] > 100)]))
print("memory_util_pct invalide :", len(df[(df["memory_util_pct"] < 0) | (df["memory_util_pct"] > 100)]))

lignes dupliquées : 0
resource_id dupliqué : 0
cost_usd <= 0 : 0
usage_quantity <= 0 : 0
cpu_util_pct invalide : 0
memory_util_pct invalide : 0


## 5. Colonnes finales

In [5]:
# cost_rounded = arrondi de cost_usd, cost_inr = conversion en roupies (x83) -> redondantes
df = df.drop(columns=["cost_rounded", "cost_inr"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   resource_id        1000 non-null   str           
 1   service            1000 non-null   str           
 2   usage_quantity     1000 non-null   float64       
 3   usage_unit         1000 non-null   str           
 4   region             1000 non-null   str           
 5   cpu_util_pct       1000 non-null   float64       
 6   memory_util_pct    1000 non-null   float64       
 7   net_in_bytes       1000 non-null   int64         
 8   net_out_bytes      1000 non-null   float64       
 9   usage_start        1000 non-null   datetime64[us]
 10  usage_end          1000 non-null   datetime64[us]
 11  cost_per_quantity  1000 non-null   float64       
 12  cost_usd           1000 non-null   float64       
 13  duree_jours        1000 non-null   float64       
dtypes: datetime64[us](2)

## 6. Sauvegarde

Le CSV stocke les dates en texte : il faudra repasser `parse_dates` à la relecture. Le dossier `data/processed/` doit déjà exister.

In [6]:
output_path = "../data/processed/gcp_costs_clean.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")

# reload check: CSV stores dates as text, parse_dates restores the dtype
df_reloaded = pd.read_csv(output_path, parse_dates=["usage_start", "usage_end"])
print("Reload OK:", df_reloaded.shape == df.shape)

Saved 1000 rows to ../data/processed/gcp_costs_clean.csv
Reload OK: True


## Synthèse

- 15 colonnes en entrée, 14 après nettoyage : suppression de `cost_rounded` et `cost_inr`, qui ne faisaient que répéter `cost_usd` (arrondi, ou converti en roupies)
- `usage_start` et `usage_end` sont maintenant de vraies dates exploitables (plus du texte). Ajout de `duree_jours` = durée entre les deux
- Contrôles : aucun doublon, aucun coût ou quantité négatif, utilisations toutes entre 0 et 100
- Sortie : `data/processed/gcp_costs_clean.csv`